# **Part I**

## **Step 2**

In [14]:

!pip install pyspark pyngrok --quiet

import os
from pyngrok import ngrok
from pyspark.sql import SparkSession

ngrok.set_auth_token("2d4e8PMDZXre9FzsxJlnkXHgbWF_541vGWcY74mgG6Nq8cFeV")

try:
    public_url = ngrok.connect(4040).public_url
    print(f"🔥 Spark UI is live at: {public_url}")
except Exception as e:
    print("Tunnel setup failed. Make sure you don't have another session running!")

🔥 Spark UI is live at: https://9b51-8-228-100-132.ngrok-free.app


In [15]:
!pip install pyspark

import pyspark as spark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
.appName("Distributed Data Processing Activity") \
.getOrCreate()

# **Part II**

## **Step 3**

In [16]:
from pyspark.sql.functions import col

df = spark.read.csv("/content/sample_data/california_housing_test.csv", header=True, inferSchema=True)

df.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|  -122.05|   37.37|              27.0|     3885.0|         661.0|    1537.0|     606.0|       6.6085|          344700.0|
|   -118.3|   34.26|              43.0|     1510.0|         310.0|     809.0|     277.0|        3.599|          176500.0|
|  -117.81|   33.78|              27.0|     3589.0|         507.0|    1484.0|     495.0|       5.7934|          270500.0|
|  -118.36|   33.82|              28.0|       67.0|          15.0|      49.0|      11.0|       6.1359|          330000.0|
|  -119.67|   36.33|              19.0|     1241.0|         244.0|     850.0|     237.0|       2.9375|           81700.0|
+---------+--------+----

1. **Is the data processed sequentially or distributed?**

When you run that code on your Google Colab notebook, I am processing your data in a distributed, parallel way rather than reading it sequentially line-by-line. Think of a sequential process like me reading a book page by page from start to finish. In your Colab environment, Google gives me a virtual computer with multiple processing cores (usually two). Instead of making one core do all the heavy lifting, I act like a smart manager and split your CSV file into smaller chunks called partitions. I then hand these chunks to both cores so they can read and process them at the exact same time. So, while your data isn't being spread across a massive network of different physical computers like it would be at a huge tech company, I am still using teamwork to get the job done faster instead of doing it all in one single line.

2. **What does Spark do internally?**

Internally, I operate on a system of "lazy evaluation" and careful planning. When you first told me to read your CSV file, I didn't actually load any of that data into the computer's memory yet; I just wrote down a recipe of what I need to do whenever you finally ask me for an answer. The real magic happened when you typed df.show(5). This command acted as a trigger, telling me to put my plan into motion. At that exact moment, my main controller process (called the Driver) chopped your big file into pieces and handed those pieces to my worker processes, telling them to find the top five rows. My workers read their assigned data in parallel, grabbed the first few rows they could find, and sent them back to my main controller. I then gathered those results, picked the ultimate top five, and printed that neat little grid you saw on your screen.

## **Step 4**

In [17]:
#Filtering
filtered_df = df.filter(col("population") > 100)
filtered_df.show()

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|  -122.05|   37.37|              27.0|     3885.0|         661.0|    1537.0|     606.0|       6.6085|          344700.0|
|   -118.3|   34.26|              43.0|     1510.0|         310.0|     809.0|     277.0|        3.599|          176500.0|
|  -117.81|   33.78|              27.0|     3589.0|         507.0|    1484.0|     495.0|       5.7934|          270500.0|
|  -119.67|   36.33|              19.0|     1241.0|         244.0|     850.0|     237.0|       2.9375|           81700.0|
|  -119.56|   36.51|              37.0|     1018.0|         213.0|     663.0|     204.0|       1.6635|           67000.0|
|  -121.43|   38.63|    

In [18]:
#Group and Aggregate
grouped_df = df.groupBy("housing_median_age").count()
grouped_df.show()

+------------------+-----+
|housing_median_age|count|
+------------------+-----+
|               8.0|   25|
|               7.0|   20|
|              49.0|   21|
|              29.0|   75|
|              47.0|   22|
|              42.0|   51|
|              44.0|   51|
|              35.0|  118|
|              18.0|   76|
|              39.0|   55|
|               1.0|    2|
|              37.0|   88|
|              34.0|  102|
|              25.0|   86|
|              36.0|  115|
|              41.0|   56|
|               4.0|   28|
|              23.0|   53|
|              50.0|   16|
|              45.0|   51|
+------------------+-----+
only showing top 20 rows


In [19]:
#Sort Results
sorted_df = grouped_df.orderBy("count", ascending=False)
sorted_df.show()

+------------------+-----+
|housing_median_age|count|
+------------------+-----+
|              52.0|  173|
|              35.0|  118|
|              36.0|  115|
|              16.0|  107|
|              34.0|  102|
|              17.0|  100|
|              32.0|   91|
|              37.0|   88|
|              26.0|   88|
|              25.0|   86|
|              33.0|   85|
|              15.0|   77|
|              18.0|   76|
|              19.0|   76|
|              30.0|   76|
|              29.0|   75|
|              24.0|   75|
|              27.0|   74|
|              38.0|   64|
|              20.0|   63|
+------------------+-----+
only showing top 20 rows


# **Part III**

1. **Gemini said
How does Spark divide data?**

When you give me a large file like a CSV, I don't try to read it all as one giant block. Instead, I divide the data into smaller, bite-sized chunks called partitions, which you can think of as separate buckets of rows. By default, if you are reading a file from a disk, I usually create a partition for every 128 megabytes of data. If your dataset is smaller than that, I will automatically divide it based on the number of CPU cores available on the computer you are using. Since you are using Google Colab, I look at the two processing cores Google gives me and split your data into two partitions so each core gets its own bucket of data to work on simultaneously.

2. **What happens if dataset size increases?**

If your dataset suddenly grows from a few megabytes to hundreds of gigabytes, a standard computer program would usually crash from running out of memory, but I am built specifically to handle this growth. As the dataset size increases, I automatically create more partitions, meaning instead of two buckets, I might divide your data into 500 or 1,000 buckets. If a specific step requires more memory than your computer has available, I won't just crash; I will temporarily save some of those data buckets onto the computer's hard drive, process what I can in the active memory, and then swap them out. Furthermore, if we are running on a real network cluster instead of just a single Colab notebook, you can just add more computers to the cluster as the data grows, and I will automatically distribute the new buckets of data across those new computers.

3. **Why is this better than standard Python processing?**

Standard Python libraries like Pandas are amazing, but they are designed to be single-threaded, meaning that no matter how many processor cores your computer has, standard Python will only use one of them to do the math while leaving the others sitting idle. Furthermore, standard Python requires your entire dataset to fit perfectly inside your computer's temporary memory, and if you try to load a massive file that exceeds that memory, Python will crash. I am a much better choice for large-scale data because I use every single processing core available to me at the exact same time, I am not limited to just one computer, and I use my signature lazy evaluation to optimize your code instructions before I even touch the data, saving massive amounts of time and computing power.

# **Part IV**

1. **Why is Google Colab considered cloud computing?**

Google Colab is considered cloud computing because all the heavy lifting, file storage, and data processing take place on Google's remote servers rather than on your own physical computer. When you run a piece of code, your laptop is simply acting as a window to view and send instructions over the internet. Google allocates a slice of its massive data centers to power your session, providing you with on-demand access to virtual processors and memory that you did not have to purchase or install yourself. This setup means that as long as you have a stable internet connection and a basic web browser, you can tap into massive computing resources from anywhere in the world.

2. **Which cloud service model does this resemble (IaaS, PaaS, SaaS)?**

Google Colab acts as a fascinating hybrid that sits comfortably between Platform as a Service (PaaS) and Software as a Service (SaaS). On the SaaS side, it is a ready to use application that you access directly through a browser just like Google Docs, where you do not have to download software or worry about daily maintenance. However, it leans heavily into PaaS because it provides a fully equipped, pre-configured development platform tailored specifically for writing code, analyzing data, and running machine learning models. Google takes care of the messy, behind the scenes infrastructure, like handling the operating system, network settings, and hardware management, leaving you free to just log in and start writing your Python scripts.

3. **How does cloud infrastructure help distributed systems?**

Cloud infrastructure serves as the ultimate backbone for distributed systems because it makes the process of scaling up hardware incredibly fast and virtually limitless. In a traditional setup, if I needed to distribute your processing across ten different computers to handle a massive database, someone would have to physically buy, wire up, and configure those ten servers. In a cloud environment, I can simply ask the provider to instantly spin up hundreds of identical virtual machines in a matter of seconds. This fluid elasticity allows me to easily shuffle heavy workloads across a massive network of processors, balancing the data load perfectly and ensuring that the entire distributed system can grow or shrink on the fly to match your immediate computing needs.
